# Notebook 03 - Inflation analysis (no inflation in B&F)

Decomposes the oil-shock response into a price and a quantity part and computes the
Domar-weighted mean price (the only 'inflation' object in the static B&F model).

CAVEAT: the `eta_price_insensitivity` helper is intentionally a no-op for the fixed-L
B&F model (there is no eta parameter here); the real eta-variance result lives in the
main BeyondHulten package. It is kept only to document the negative finding.

In [ ]:
# --- Project setup (robust path resolution) ---
# @__DIR__ resolves to this notebook's directory; we anchor on the package root.
using LinearAlgebra, Statistics, Printf, DelimitedFiles

const NOTEBOOK_DIR = @__DIR__
const REP_DIR = joinpath(NOTEBOOK_DIR, "..")          # bf_replication/
cd(REP_DIR)
include(joinpath(REP_DIR, "src", "BFReplication.jl"))
using .BFReplication
using .BFReplication.DataLoader
using .BFReplication.BFModel
using .BFReplication.InflationAnalysis

const DATA_DIR = joinpath(REP_DIR, "..", "Replication Files", "GDP Simulatin -- 88 Sector")
const RESULTS_DIR = joinpath(REP_DIR, "data", "results")
mkpath(RESULTS_DIR)

println("Project dir : ", REP_DIR)
println("Data dir    : ", DATA_DIR)
println("Results dir : ", RESULTS_DIR)

# --- Load data ---
data = load_bf_data(joinpath(DATA_DIR, "BFdata.csv"); year=1980)
describe_data(data)

## Baseline vs oil shock: price vs quantity

In [ ]:
# analyze_price_vs_quantity takes (BFParameters, BFParameters, sector)
A_base = ones(data.N)
A_shock = ones(data.N); A_shock[7] = 0.7
p_base = BFParameters(A_base, data.Ω, data.α, data.β, data.L, 0.5, 0.0001, 0.9)
p_shock = BFParameters(A_shock, data.Ω, data.α, data.β, data.L, 0.5, 0.0001, 0.9)

pq = analyze_price_vs_quantity(p_base, p_shock, 7)
println("oil-shock log price change    = $(pq.mean_price_change)")
println("oil-shock log quantity change = $(pq.real_gdp_change)")

## Inflation measures (Domar-weighted mean price)

In [ ]:
# compute_inflation_measures takes (sol_baseline, sol_shocked, params)
sol_base = BFModel.compute_equilibrium(p_base)
sol_shock = BFModel.compute_equilibrium(p_shock)

measures = compute_inflation_measures(sol_base, sol_shock, p_shock)
println("mean log price change (Domar-weighted) = $(measures.mean_price_change)")

## eta price insensitivity + network decomposition (see caveat above)

In [ ]:
# eta_price_insensitivity takes (data, sector)
eta = eta_price_insensitivity(data, 7)
println("price CV to eta = $(eta.cpi_cv)   (no eta in fixed-L B&F -> no-op)")

# network_price_decomposition takes (params)
net = network_price_decomposition(p_shock)
println("network decomposition: ", net)